# Review of Claude's recent7 and first-93-feature candidates

This report displays the completed fixed-recipe audit. The independent reproduction ran the supplied generation script; all historical fitting results and verification are preserved separately.

In [1]:
from pathlib import Path
import json
import pandas as pd

REVIEW_DIR = Path.cwd() / "claude_candidates_review_results"
if not REVIEW_DIR.is_dir():
    REVIEW_DIR = Path('/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/claude_candidates_review_results')
summary = pd.read_csv(REVIEW_DIR / "summary.csv")
verification = json.loads((REVIEW_DIR / "independent_verification.json").read_text())
assert verification["all_auc_values_independently_recalculated"]
assert verification["all_group_means_independently_recalculated"]
assert verification["original_sources_unchanged"]
assert all(v["reproduced_within_1e_7"] for v in verification["bare_script_reproduction"].values())
display(summary.loc[summary["group"].isin(["all_15", "later_seven", "latest"]),
                    ["group", "model", "mean_auc", "mean_delta_vs_submitted_recipe"]])
print("Verified reproduction within 1e-7; exact equality is not established.")


Verified reproduction within 1e-7; exact equality is not established.


,group,model,mean_auc,mean_delta_vs_submitted_recipe
0,all_15,claude_fixed_mask_control,0.955496,0.000525
1,all_15,claude_local_features,0.945724,-0.009248
2,all_15,claude_recent7,0.955913,0.000941
3,all_15,lightgbm_leaf_5,0.955732,0.000761
4,all_15,lightgbm_leaf_50,0.958422,0.003451
5,all_15,lightgbm_leaf_blend,0.957620,0.002649
6,all_15,submitted_recipe,0.954972,0.000000
14,later_seven,claude_fixed_mask_control,0.926432,0.000794
15,later_seven,claude_local_features,0.909940,-0.015698
16,later_seven,claude_recent7,0.926059,0.000421


## Findings from the completed review

**12 October 2026.** The student requested review of two prepared Claude candidates and supplied their new regeneration script, `claude_experiments/scripts/make_recent7_and_local_features.py`, together with original cloud experiment scripts. This review reproduces and evaluates those supplied fixed recipes. It introduces no new candidate settings, automatic tuning, feature design, blend weights or final-model selection.

### What the two candidates do

Both average a 200-tree XGBoost and a 150-tree LightGBM equally. They retain the prior low-regularisation settings, seed 42 and two CPU threads. They fit all **31,235 confirmed labels** through step 35. Unknown labels receive no invented answers and are excluded from supervised fitting. The test features are used for prediction only; no external data or hidden test labels enter fitting.

**Recent7** applies `0.5 ** ((training_end - time_step) / 7)` to each known fitting row, then divides by the mean weight. Thus an example seven steps older receives half as much weight, and the total fitting weight remains equal to the number of rows. This retains every confirmed row rather than training only on the final seven periods. At the full fit, weights range from **0.129805** to **3.762162**. The weight concentration corresponds to an effective sample size of about **15,107**, calculated as `(sum weights)^2 / sum(weights^2)`; it is not a count of dropped examples. XGBoost excludes six previously selected time-correlated features; LightGBM uses all 165 features.

**First 93 features** fits both components using `feat_1` through `feat_93`, with natural equal row weights. All settings and confirmed fitting rows remain the same. This is an experiment in removing the remaining feature block. The supplied files and source establish column use; external historical-event narratives are not treated as competition evidence.

### Actual reproduction and verification

The original script was copied unchanged and executed in isolated folders with a read-only link to the supplied Data Set folder. All original CSVs, data and scripts remained unchanged. Both regenerated files contain the correct **15,329 sample indices**, aligned prediction scores and valid probabilities. A second direct script invocation in a fresh interpreter confirmed the same result.

The reproduction is close but **not numerically or byte-for-byte exact on this machine**. Recent7 differs on 119 rows, with maximum absolute probability difference **2.980232e-08**; the first-93-feature candidate differs on 516 rows, with maximum difference **5.960464e-08**. Both reproduce within 1e-7 and have rank correlation effectively 1 with the originals. These are tiny floating-point differences. The original assertion of a difference of exactly 0 on every row is not confirmed by this independent run. No replacement of the original submission files was made.

### Existing chronological comparisons

The audit refitted only the supplied fixed recipes on the same 15 previously selected fourteen-period checks: train through k=7..21 and check k+1..k+14. It also fitted Claude's unweighted fixed-mask control to distinguish recency weighting from differences in time-feature exclusions. Previously verified submitted-recipe and LightGBM reference probabilities were reused only after their checking identifiers, labels and time steps exactly matched. Every AUC and group mean was independently recalculated.

| Recipe | All 15 mean | Later seven mean | Previously chosen four | Latest check |
|---|---:|---:|---:|---:|
| Submitted recipe | 0.954972 | 0.925638 | 0.935353 | 0.916956 |
| Claude fixed-mask control | 0.955496 | 0.926432 | 0.936000 | 0.916956 |
| Claude recent7 | 0.955913 | 0.926059 | 0.935922 | 0.912482 |
| Claude first 93 features | 0.945724 | 0.909940 | 0.923188 | 0.900551 |
| LightGBM 5/50 blend | 0.957620 | 0.930209 | 0.940213 | 0.917834 |
| LightGBM leaf 50 | 0.958422 | 0.931137 | 0.942234 | 0.916709 |

Recent7 changes the all-15 mean by **+0.000941** against the submitted recipe, but only **+0.000417** against the matching fixed-mask control. Its later-seven mean changes by **-0.000373** against that control, and its latest check falls by **-0.004473** against the submitted recipe. It wins **3 of seven later checks** against the submitted recipe and **3 of seven** against the LightGBM blend. Thus the available evidence supports only a small, mixed development gain.

The first-93-feature candidate loses to the submitted recipe and the LightGBM blend on **all 15 checks**. Its later-seven mean falls by **-0.015698** against the submitted recipe. Its public result could still differ, but it is an exploratory feature-removal submission with weaker local evidence.

### Validation limitation found in the original cloud scripts

The six excluded columns were selected using steps **1–32**, then hardcoded for all historical checks. That selection includes future rows relative to the earlier checking cutoffs, whereas the submitted-recipe reference recomputes the time filter within each fitting period. The fixed mask differs from the training-only reference in **14 of 15** checks. The comparison table faithfully audits the original cloud recipes; the fixed-mask control isolates recency weighting but does not repair that historical selection limitation. At the latest cutoff 21 the masks coincide, and recent7's worse latest result therefore is not explained by that mask mismatch.

The final fit through 35 has no corresponding test-data lookahead: the exclusion was selected entirely from earlier supplied training data. This caveat concerns the interpretation of the original development checks, not a finding that the final generator trained on test answers. All fifteen checking windows also overlap and have been repeatedly used for development; their results do not provide independent held-out confidence.

### What a public submission can establish

A public ROC-AUC above the existing 0.94701 would show that the submitted recipe ranked the public subset better. A lower score would favour the existing recipe on that subset. An aggregate result cannot locate the improvement in a particular time period, prove an external event explanation, determine the hidden private score, or rule out all uses of the removed feature block. No calibrated expected Kaggle score or evidence of 0.96+ follows from these local comparisons. The verified submitted public best remains **0.94701**; both Claude candidate public scores remain unknown.

**AI-assisted assessment:** the 5/50 LightGBM blend has higher all-15, later-seven, previously chosen-four and latest AUC than recent7, although recent7 wins nine individual checks overall. The blend remains the better-supported next score-improvement candidate among these three. The first-93-feature candidate has a distinct diagnostic purpose if the student deliberately chooses that use of a submission. Final selection remains with the student.

**AI assistance record:** Claude supplied the candidate recipes and generation code. OpenAI Codex inspected the code, independently regenerated predictions, refitted the existing fixed recipes on the existing checking windows, checked controls and artifacts, and drafted this review. The student remains responsible for understanding and defending the modeling choices, disclosing assistance, choosing submissions and interpreting results. No Kaggle upload occurred.

The supplied generator is complete code for fitting these two candidates from the provided competition files. This review notebook displays recorded evidence; it is not a new model or an autonomous search. Results and the unchanged source snapshot are in `claude_candidates_review_results/`.
